# qwen2.5 в коллабе

сравнение модели 8 и 4 бита, посмотреть ответы позволит сокрость и потребление врама

среда исполнения --> сменить среду выполнения --> гп т4 --> сейв & подключение
по умолчанию стоит 3B, все три режима и по 3 повтора


In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers", "accelerate", "bitsandbytes",
    "huggingface-hub", "pandas", "matplotlib", "psutil",
]) # пока без фикс версий потестить потом везде латест взять и записать
print("ready")


## что гоняем

MODEL_SIZE — размер модели, можно выбрать 1.5B, 3B или 7B. 7B в fp16 на т4 может не влезть, пока лучше 3B

REPEATS — сколько раз гоняем каждый тест. если 3, то каждый запрос отправится 3 раза в каждом режиме
нужно чтоб усреднить скорость, она может немного прыгать. для быстрой проверки можно поставить 1
в процентах прохождения один тест всё равно считается один раз

MAX_NEW_TOKENS — сколько токенов максимум модель может написать в ответе. токен это кусочек текста, не обязательно целое слово
128 оставил для коротких ответов, если нужны объяснения или большой json можно поставить 512
это потолок, модель может закончить раньше. если упрётся в лимит, ответ обрежется, это будет в hit_token_limit

fp16, int8 и nf4 гоняются подряд, выбирать режим не надо


In [ ]:
# @title настройки
MODEL_SIZE = "3B" # @param ["1.5B", "3B", "7B"]
REPEATS = 3 # @param {type:"integer"}
MAX_NEW_TOKENS = 128 # @param {type:"integer"}

SEED = 42
SYSTEM_PROMPT = "Ты полезный помощник. Следуй инструкции пользователя точно."

assert MODEL_SIZE in {"1.5B", "3B", "7B"}
assert isinstance(REPEATS, int) and REPEATS >= 1
assert isinstance(MAX_NEW_TOKENS, int) and MAX_NEW_TOKENS >= 1
MODEL_ID = f"Qwen/Qwen2.5-{MODEL_SIZE}-Instruct"
PRECISIONS = ["fp16", "int8", "nf4"]
print(MODEL_ID, PRECISIONS)


## загрузка тестов

сюда закинуть prompts.jsonl, без файла дальше не запустится
один тест на строку, без общего списка и запятых между строками

exact — точный ответ, регистр и пробелы по краям не важны
one_of — любой вариант из expected
json — сравнить json с эталоном
manual — потом оценить руками по rubric, в автоматический процент не идёт


In [ ]:
"""читаем тесты и проверяем ответы"""

import json
from pathlib import Path


def normalize(text):
    """убрать пробелы по краям и привести к одному регистру"""
    return text.strip().casefold()


def canonical_json(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True, allow_nan=False)


def check_answer(test, answer):
    kind = test["check"]
    if kind == "manual":
        return None  # это потом посмотреть руками
    if kind == "exact":
        return normalize(answer) == normalize(test["expected"])
    if kind == "one_of":
        return normalize(answer) in {normalize(item) for item in test["expected"]}
    if kind == "json":
        try:
            def reject_constant(value):
                raise ValueError(f"такое значение в JSON не подходит: {value}")

            def unique_keys(pairs):
                result = {}
                for key, value in pairs:
                    if key in result:
                        raise ValueError(f"ключ в JSON повторяется: {key}")
                    result[key] = value
                return result

            actual = json.loads(answer, parse_constant=reject_constant,
                                object_pairs_hook=unique_keys)
            return canonical_json(actual) == canonical_json(test["expected"])
        except (ValueError, TypeError):
            return False
    raise ValueError(f"не знаю такую проверку: {kind}")


def validate_tests(tests):
    if not tests:
        raise ValueError("тестов нет, надо добавить")
    seen = set()
    for index, test in enumerate(tests, 1):
        prefix = f"тест {index}"
        if not isinstance(test, dict):
            raise ValueError(f"{prefix}: тут должен быть JSON-объект.")
        for key in ["id", "category", "prompt", "check"]:
            if not isinstance(test.get(key), str) or not test[key].strip():
                raise ValueError(f"{prefix}: надо заполнить строку {key!r}.")
        identifier = test["id"]
        if not all(c.isalnum() or c in "_-" for c in identifier):
            raise ValueError(f"{prefix}: в id нужны только буквы, цифры, _ или -.")
        if identifier in seen:
            raise ValueError(f"{prefix}: такой id уже есть: {identifier!r}.")
        seen.add(identifier)
        kind = test["check"]
        if kind not in {"exact", "one_of", "json", "manual"}:
            raise ValueError(f"{prefix}: в check поставить exact, one_of, json или manual.")
        if kind == "manual":
            if not isinstance(test.get("rubric"), str) or not test["rubric"].strip():
                raise ValueError(f"{prefix}: для manual надо написать критерии в rubric.")
            continue
        if "expected" not in test:
            raise ValueError(f"{prefix}: нет эталонного ответа в expected.")
        expected = test["expected"]
        if kind == "exact" and not isinstance(expected, str):
            raise ValueError(f"{prefix}: для exact ответ в expected надо писать строкой.")
        if kind == "one_of" and not (
            isinstance(expected, list) and expected
            and all(isinstance(item, str) and item.strip() for item in expected)
        ):
            raise ValueError(f"{prefix}: для one_of в expected нужен список вариантов ответа.")
        if kind == "json":
            try:
                canonical_json(expected)
            except (TypeError, ValueError) as error:
                raise ValueError(f"{prefix}: эталон в expected не получается записать в JSON.") from error
    return tests


def load_tests(path):
    tests = []
    with Path(path).open(encoding="utf-8-sig") as handle:
        for line_number, line in enumerate(handle, 1):
            if not line.strip():
                continue
            try:
                tests.append(json.loads(line))
            except ValueError as error:
                raise ValueError(f"{path}, строка {line_number}: что-то не так с JSON: {error}") from error
    return validate_tests(tests)



from google.colab import files

# чтоб случайно не запустить старые тесты если новую загрузку отменил
TESTS = None
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("надо загрузить один файл с тестами, в формате .jsonl")
uploaded_name, uploaded_content = next(iter(uploaded.items()))
if not uploaded_name.lower().endswith(".jsonl"):
    raise ValueError("нужен файл .jsonl")
temporary_dataset = Path("/content/qwen25_uploaded_prompts.jsonl")
temporary_dataset.write_bytes(uploaded_content)
TESTS = load_tests(temporary_dataset)
manual_count = sum(test["check"] == "manual" for test in TESTS)
print(f"тестов: {len(TESTS)}, автоматом: {len(TESTS) - manual_count}, руками: {manual_count}")
print(f"всего замеров: {len(TESTS) * REPEATS * len(PRECISIONS)}")


## железо и папка для результатов

проверить что включена gpu и записать настройки, версии библиотек и инфу о железе
имя папки само соберётся из времени запуска, размера модели, повторов, лимита ответа и хеша настроек

всё пока лежит внутри коллаба, в конце скачается архив. drive не нужен
если сеанс отключится и коллаб удалит файлы, продолжить по ним уже не выйдет


In [ ]:
import gc
import hashlib
import importlib.metadata
import platform
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import psutil
import torch
from huggingface_hub import HfApi
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
from transformers.generation.streamers import BaseStreamer

if not globals().get("TESTS"):
    raise RuntimeError("сначала загрузи файл с тестами в ячейке выше")

if not torch.cuda.is_available():
    raise RuntimeError("GPU не включена. в коллабе выбери: среда выполнения → сменить среду выполнения → GPU T4.")
GPU = torch.cuda.get_device_properties(0)
GPU_CAPABILITY = torch.cuda.get_device_capability(0)
GPU_INFO = subprocess.check_output([
    "nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"
], text=True).strip()
print(GPU_INFO)
print("CUDA:", torch.version.cuda, "PyTorch:", torch.__version__)

ROOT = Path("/content/qwen25_quant_eval")

# фиксируем одну версию весов, чтоб во всех режимах была одна модель
MODEL_REVISION = HfApi().model_info(MODEL_ID).sha
METADATA = {
    "notebook_version": 3,
    "model_id": MODEL_ID, "model_revision": MODEL_REVISION,
    "precisions": PRECISIONS, "repeats": REPEATS, "seed": SEED,
    "generation": {"do_sample": False, "max_new_tokens": MAX_NEW_TOKENS,
                   "num_beams": 1, "repetition_penalty": 1.0, "use_cache": True},
    "system_prompt": SYSTEM_PROMPT, "tests": TESTS,
    "backend": "transformers", "attention": "sdpa", "batch_size": 1,
    "compute_dtype": "float16", "nf4_double_quant": True,
    "gpu": GPU_INFO, "gpu_capability": list(GPU_CAPABILITY),
    "python": platform.python_version(), "cuda": torch.version.cuda,
    "packages": {name: importlib.metadata.version(name) for name in
                 ["torch", "transformers", "accelerate", "bitsandbytes", "huggingface-hub", "pandas", "psutil"]},
    "measurement": {
        "tokens_per_second": "all generated token IDs (including EOS) / generation wall time",
        "ttft_seconds": "time from generate call to first token callback; excludes tokenization",
        "peak_vram_gib": "PyTorch peak allocated CUDA memory during generation, including model",
        "ram_rss_peak_gib": "process RSS sampled every 50 ms during generation, including model",
        "warmup": "one unmeasured generation of up to 16 tokens per precision per session",
        "streamer": "token callbacks used for all measurements; overhead included",
    },
}
FINGERPRINT = hashlib.sha256(json.dumps(METADATA, sort_keys=True, ensure_ascii=False).encode()).hexdigest()[:16]
# при повторе этой ячейки с теми же настройками используем папку текущего прогона
if globals().get("ACTIVE_FINGERPRINT") != FINGERPRINT or not globals().get("RUN_DIR") or not RUN_DIR.exists():
    STARTED_AT = datetime.now(timezone.utc)
    RUN_NAME = (f"{STARTED_AT.strftime('%Y%m%d_%H%M%S_%f')}_"
                f"{MODEL_SIZE}_fp16-int8-nf4_r{REPEATS}_t{MAX_NEW_TOKENS}_{FINGERPRINT[:8]}")
    RUN_DIR = ROOT / RUN_NAME
    ACTIVE_FINGERPRINT = FINGERPRINT
RAW_DIR = RUN_DIR / "raw_logs"
RAW_DIR.mkdir(parents=True, exist_ok=True)
METADATA["run_name"] = RUN_NAME
METADATA["started_at_utc"] = STARTED_AT.isoformat()

def save_json(path, value):
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")
    temporary.replace(path)

metadata_path = RUN_DIR / "metadata.json"
if metadata_path.exists():
    assert json.loads(metadata_path.read_text(encoding="utf-8")) == METADATA
else:
    save_json(metadata_path, METADATA)
print("сохраняю сюда:", RUN_DIR)


## загрузка модели и замеры

модель целиком на gpu, сначала короткий прогрев, потом тесты
скорость считаю как токены делённые на время генерации, первый токен отдельно
скачивание модели и подготовка запроса сюда не входят
врам через pytorch, цифра может отличаться от nvidia-smi


In [ ]:
from copy import deepcopy
from threading import Event, Thread

GiB = 1024 ** 3

def load_model(precision):
    if precision == "int8" and GPU_CAPABILITY < (7, 5):
        raise RuntimeError("для int8 нужна T4 или новее. выбери GPU T4 в настройках коллаба.")
    options = dict(revision=MODEL_REVISION, device_map={"": 0},
                   dtype=torch.float16, attn_implementation="sdpa")
    if precision == "int8":
        options["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)
    elif precision == "nf4":
        options["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
        )
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **options)
    model.eval()
    config = deepcopy(model.generation_config)
    config.do_sample = False
    config.temperature = None
    config.top_p = None
    config.top_k = None
    config.num_beams = 1
    config.repetition_penalty = 1.0
    config.use_cache = True
    config.max_new_tokens = MAX_NEW_TOKENS
    return model, config

TOKENIZER = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
if TOKENIZER.pad_token_id is None:
    TOKENIZER.pad_token = TOKENIZER.eos_token

def make_inputs(prompt):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": prompt}]
    text = TOKENIZER.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return TOKENIZER(text, return_tensors="pt").to("cuda:0")

class FirstTokenTimer(BaseStreamer):
    def __init__(self):
        self.is_prompt = True
        self.first_token_time = None

    def put(self, value):
        if self.is_prompt:
            self.is_prompt = False
        elif self.first_token_time is None:
            torch.cuda.synchronize()
            self.first_token_time = time.perf_counter()

    def end(self):
        pass

class RamSampler:
    def __enter__(self):
        self.process = psutil.Process()
        self.peak = self.process.memory_info().rss
        self.stop = Event()
        def sample():
            while not self.stop.wait(0.05):
                self.peak = max(self.peak, self.process.memory_info().rss)
        self.thread = Thread(target=sample, daemon=True)
        self.thread.start()
        return self

    def __exit__(self, *args):
        self.stop.set()
        self.thread.join()
        self.peak = max(self.peak, self.process.memory_info().rss)

def measure(model, config, prompt):
    inputs = make_inputs(prompt)
    timer = FirstTokenTimer()
    set_seed(SEED)
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    with RamSampler() as ram, torch.inference_mode():
        started = time.perf_counter()
        output = model.generate(**inputs, generation_config=config,
                                pad_token_id=TOKENIZER.pad_token_id, streamer=timer)
        torch.cuda.synchronize()
        finished = time.perf_counter()
    generated = output[0, inputs["input_ids"].shape[1]:]
    elapsed = finished - started
    answer = TOKENIZER.decode(generated, skip_special_tokens=True).strip()
    return {
        "answer": answer, "input_tokens": int(inputs["input_ids"].shape[1]),
        "output_tokens": int(generated.numel()), "generation_seconds": elapsed,
        "tokens_per_second": int(generated.numel()) / elapsed,
        "ttft_seconds": None if timer.first_token_time is None else timer.first_token_time - started,
        "peak_vram_gib": torch.cuda.max_memory_allocated() / GiB,
        "peak_reserved_vram_gib": torch.cuda.max_memory_reserved() / GiB,
        "ram_rss_peak_gib": ram.peak / GiB,
        "hit_token_limit": int(generated.numel()) >= MAX_NEW_TOKENS,
    }

print("можно запускать")


## запуск

ответы сразу пишутся в json, если какой-то режим упал ошибка тоже сохранится
если файлы ещё есть в текущем сеансе, можно повторить эту ячейку, готовое пропустится


In [ ]:
if not globals().get("TESTS"):
    raise RuntimeError("сначала загрузи файл с тестами")
# если поменял настройки или файл, сначала ещё раз выполнить ячейку с папкой результатов
if (MODEL_ID != METADATA["model_id"] or REPEATS != METADATA["repeats"]
        or MAX_NEW_TOKENS != METADATA["generation"]["max_new_tokens"]
        or TESTS != METADATA["tests"]):
    raise RuntimeError("настройки или тесты поменялись, выполни снова ячейки с папкой и загрузкой модели")

for precision in PRECISIONS:
    pending = [(test, repeat) for test in TESTS for repeat in range(1, REPEATS + 1)
               if not (RAW_DIR / f"{precision}_{test['id']}_{repeat}.json").exists()]
    if not pending:
        print(f"{precision}: уже всё есть, пропускаю.")
        continue
    print(f"\nгружу {MODEL_ID}, режим {precision}. осталось замеров: {len(pending)}")
    model = None
    try:
        gc.collect()
        torch.cuda.empty_cache()
        model, config = load_model(precision)
        footprint = model.get_memory_footprint() / GiB
        warmup_inputs = make_inputs(TESTS[0]["prompt"])
        with torch.inference_mode():
            model.generate(**warmup_inputs, generation_config=config,
                           max_new_tokens=16, pad_token_id=TOKENIZER.pad_token_id)
        del warmup_inputs
        torch.cuda.synchronize()
        for index, (test, repeat) in enumerate(pending, 1):
            result = measure(model, config, test["prompt"])
            record = {
                "experiment_id": RUN_NAME, "settings_fingerprint": FINGERPRINT,
                "timestamp_utc": datetime.now(timezone.utc).isoformat(),
                "model_id": MODEL_ID, "model_revision": MODEL_REVISION,
                "precision": precision, "backend": "transformers", "seed": SEED,
                "gpu": GPU_INFO, "test_id": test["id"], "category": test["category"],
                "prompt": test["prompt"], "expected": test.get("expected"), "check": test["check"],
                "rubric": test.get("rubric"), "repeat": repeat,
                "model_footprint_gib": footprint, **result,
                "passed": check_answer(test, result["answer"]),
            }
            save_json(RAW_DIR / f"{precision}_{test['id']}_{repeat}.json", record)
            print(f"[{index}/{len(pending)}] {test['id']} / {repeat}: "
                  f"{'руками' if record['passed'] is None else ('ок' if record['passed'] else 'не прошёл')}, "
                  f"{record['tokens_per_second']:.1f} токен/с")
        (RUN_DIR / f"error_{precision}.json").unlink(missing_ok=True)
    except Exception as error:
        save_json(RUN_DIR / f"error_{precision}.json", {
            "precision": precision, "timestamp_utc": datetime.now(timezone.utc).isoformat(),
            "error_type": type(error).__name__, "error": str(error),
        })
        print(f"режим {precision} упал: {type(error).__name__}: {error}")
        print("то что успело посчитаться, сохранено. если не хватило памяти, попробуй модель поменьше или другой режим.")
    finally:
        del model
        gc.collect()
        torch.cuda.empty_cache()

print("\nтут закончили, ниже таблица с результатами.")


## результаты

процент по первому ответу на каждый тест, скорость усредняется по повторам
ручные ответы в manual_review.csv, там вписать оценку самому

completed — сколько замеров готово, expected — сколько должно быть
если не всё готово, сравнение ещё неполное. если повтор один, разброс времени не посчитать


In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt

records = [json.loads(path.read_text(encoding="utf-8")) for path in sorted(RAW_DIR.glob("*.json"))]
if not records:
    raise RuntimeError("результатов пока нет. посмотри выше, на чём упало.")
df = pd.DataFrame(records)
df.to_csv(RUN_DIR / "answers.csv", index=False, encoding="utf-8-sig")

summary_rows = []
for precision in PRECISIONS:
    group = df[df["precision"] == precision]
    first = group[group["repeat"] == 1]
    auto = first[first["passed"].notna()]
    summary_rows.append({
        "precision": precision,
        "completed": len(group), "expected": len(TESTS) * REPEATS,
        "first_repeat_tests": len(first),
        "auto_scored_tests": len(auto),
        "manual_review_tests": len(first) - len(auto),
        "auto_pass_rate_pct": 100 * auto["passed"].astype(float).mean(),
        "tokens_per_second_mean": group["tokens_per_second"].mean(),
        "tokens_per_second_std": group["tokens_per_second"].std(),
        "ttft_seconds_mean": group["ttft_seconds"].mean(),
        "peak_vram_gib_max": group["peak_vram_gib"].max(),
        "ram_rss_peak_gib_max": group["ram_rss_peak_gib"].max(),
        "token_limit_hits": int(group["hit_token_limit"].sum()),
    })
summary = pd.DataFrame(summary_rows)
summary.to_csv(RUN_DIR / "summary.csv", index=False, encoding="utf-8-sig")
auto_first = df[(df["repeat"] == 1) & df["passed"].notna()].copy()
auto_first["passed"] = auto_first["passed"].astype(float)
by_category = auto_first.groupby(["precision", "category"])["passed"].agg(["count", "mean"])
manual_review = df[(df["repeat"] == 1) & df["passed"].isna()].copy()
manual_review["human_score"] = ""
manual_review["human_comment"] = ""
manual_review.to_csv(RUN_DIR / "manual_review.csv", index=False, encoding="utf-8-sig")
by_category.to_csv(RUN_DIR / "by_category.csv", encoding="utf-8-sig")
display(summary.round(3))
display(df[["precision", "test_id", "repeat", "answer", "passed"]])

complete = summary[summary["completed"] == summary["expected"]].set_index("precision")
if not complete.empty:
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
    for ax, column, title in zip(axes,
        ["tokens_per_second_mean", "peak_vram_gib_max", "auto_pass_rate_pct"],
        ["скорость, токен/с", "пик памяти GPU, GiB", "тесты прошли, %"]):
        complete[column].plot.bar(ax=ax, color="#4c72b0", rot=0)
        ax.set_title(title)
        ax.set_xlabel("")
        ax.set_ylim(bottom=0)
    axes[2].set_ylim(0, 100)
    fig.tight_layout()
    fig.savefig(RUN_DIR / "comparison.png", dpi=160, bbox_inches="tight")
    plt.show()

missing = int(summary["expected"].sum() - summary["completed"].sum())
print("все замеры готовы" if not missing else f"ещё не готово замеров: {missing}. посмотри ошибки выше и запусти ещё раз.")
print("папка с результатами:", RUN_DIR)


## скачать

архив с ответами, настройками, таблицами, графиком и ошибками если были
csv открываются в excel, ручные оценки потом вписать в manual_review.csv

в репо json в results/raw_logs, таблицы в results/tables, график в figures


In [ ]:
import shutil
from google.colab import files

archive_path = shutil.make_archive(f"/content/{RUN_DIR.name}", "zip", root_dir=RUN_DIR)
files.download(archive_path)
print("архив готов:", archive_path)


## для статьи

8 и 4 бита тут через bitsandbytes, awq gptq gguf пока нет
во всех режимах одинаковые веса, запросы и настройки, обучение не делается

процент только по загруженным тестам. лишняя точка или е вместо ё могут завалить exact
если допустимы разные варианты ответа, писать их в one_of

для замера скорости нужны ещё длинные ответы, на одном слове выводы делать рано
output_tokens — длина ответа в токенах, hit_token_limit — дошёл ли он до лимита
итоговую скорость лучше сравнивать в одном сеансе

загрузка и квантизация отсюда: [qwen](https://huggingface.co/Qwen/Qwen2.5-3B-Instruct), [bitsandbytes](https://huggingface.co/docs/transformers/quantization/bitsandbytes)
